In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import transforms
from torchvision.transforms import functional as TF
from PIL import Image
import torch
from collections import Counter
import numpy as np
from torch.amp import GradScaler, autocast
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Data Loader

In [3]:
ds = load_dataset("Ar4ikov/celebA_spoof")

Resolving data files:   0%|          | 0/93 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/93 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/78 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/56 [00:00<?, ?it/s]

In [4]:
img_size = 224
random_erase_prob = 0.25  
batch_size = 32
persistent_workers = True
prefetch_factor = 4
num_workers = 4
pin_memory = True
min_lr = 1e-6
warmup_epochs = 3

In [22]:
def get_transforms(is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=random_erase_prob),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

def letterbox(img, size):
    img = img if isinstance(img, Image.Image) else Image.open(img).convert("RGB")
    img.thumbnail((size, size), Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    x = (size - img.width) // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))

    return canvas

class CelebASpoofDataset(Dataset):
    def __init__(self, data, is_train=True):
        self.data = data
        self.transform = get_transforms(is_train)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]['Filepath']
        label = 1 if self.data[idx]['Class'] == 'live' else 0

        img = letterbox(img, img_size)
        arr_img = self.transform(img)

        return img, arr_img, label, idx

In [23]:
def dataset_collate(batch):
    images  = []
    targets = []
    indicies = []
    for _, arr_img, label, idx in batch:
        images.append(arr_img)
        targets.append(label)
        indicies.append(idx)
    images  = torch.from_numpy(np.array(images)).type(torch.FloatTensor)
    targets = torch.from_numpy(np.array(targets)).long()
    indicies = torch.from_numpy(np.array(indicies)).long()
    return images, targets, indicies

In [29]:
ds_test = ds['test'].shuffle(seed=42).select(range(10_000))
test_dataset = CelebASpoofDataset(ds_test, is_train=False)

In [30]:
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size * 2,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor,
    collate_fn=dataset_collate,
)

### Model

In [9]:
model_name = "vit_base_patch16_224"
pretrained = True
dropout = 0.1
num_classes = 2

In [10]:
class ViTFaceAntiSpoofing(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        embed_dim = self.vit.num_features if hasattr(self.vit, 'num_features') else 768
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.vit(x)
        return self.classifier(features)

In [11]:
model = ViTFaceAntiSpoofing().to(device)

In [12]:
sum(p.numel() for p in model.parameters())

86194946

### Load Model

In [13]:
checkpoint = torch.load("model_final.pth", map_location=device, weights_only=True)

In [14]:
model = ViTFaceAntiSpoofing().to(device)
model.load_state_dict(checkpoint)

<All keys matched successfully>

### Test

In [15]:
model.eval()
pass

In [31]:
len(test_dataset)

10000

In [17]:
mixed_precision = True

In [ ]:
with torch.no_grad():
    all_preds = []
    all_labels = []
    all_probs = []
    all_indices = []
    
    pbar = tqdm(test_loader, desc="Testing")
    
    for images, labels, indices in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with autocast(device_type=device.type, enabled=mixed_precision):
            outputs = model(images)
        
        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # Probability of being live
        all_indices.extend(indices.numpy())

Testing:   0%|                                                                          | 0/157 [00:00<?, ?it/s]